In [1]:
from ATLAS.Analysis.Classification import *
from ATLAS.Registration.Registration import *

<frozen importlib._bootstrap>:228: RuntimeWarning: scipy._lib.messagestream.MessageStream size changed, may indicate binary incompatibility. Expected 56 from C header, got 64 from PyObject


In [2]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [3]:
""" Load CCF"""
import nrrd
from matplotlib.colors import Normalize, ListedColormap
allen_ontology = pd.read_csv('/scratchdata1/MouseBrainAtlases/Taxonomies/CCF_Ref/ccf_2017/allen_ontology.csv')
color_mapper = dict(zip(allen_ontology['name'], ['#'+str(i) for i in allen_ontology['color_hex_triplet']]))
color_mapper[np.nan] = 'k'
annotation_mapper = dict(zip(allen_ontology['acronym'], allen_ontology['name']))
reverse_annotation_mapper = {v: k for k, v in annotation_mapper.items()}
filename = '/scratchdata1/MouseBrainAtlases/Taxonomies/CCF_Ref/ccf_2017/annotation_25.nrrd'
ccf_x_window = [4,10]
ccf_x_min = int(ccf_x_window[0]*1000/25)
ccf_x_max = int(ccf_x_window[1]*1000/25)
readdata, header = nrrd.read(filename)
readdata_torch = torch.tensor(readdata.astype(int))
readdata_torch = readdata_torch[ccf_x_min:ccf_x_max,:,:]
def get_map(data,allen_ontology):
    level,label = data
    mapper = {}
    structure_id = str(allen_ontology[allen_ontology['name'] == label]['id'].values[0])
    ids = [row['id'] for i,row in allen_ontology.iterrows() if structure_id in row['structure_id_path'].split('/')]
    for id in ids:
        mapper[id] = label
    return level,label,mapper


del readdata
level_mapper = {}
Input = []
for level in sorted(allen_ontology['st_level'].unique()):
    level_mapper[level] = {row['id']:row['name'] for idx,row in allen_ontology.iterrows()}
    for label in allen_ontology[allen_ontology['st_level'] == level]['name']:
        Input.append((level,label))
        # structure_id = str(allen_ontology[allen_ontology['name'] == label]['id'].values[0])
        # ids = [row['id'] for i,row in allen_ontology.iterrows() if structure_id in row['structure_id_path'].split('/')]
        # for id in ids:
        #     level_mapper[level][id] = label
pfunc = partial(get_map, allen_ontology=allen_ontology)
with multiprocessing.Pool(30) as pool:
    for level,label,mapper in tqdm(pool.imap(pfunc, Input), total=len(Input), desc='Mapping levels'):
        level_mapper[level].update(mapper)


Mapping levels: 100%|██████████| 1327/1327 [00:09<00:00, 145.05it/s]


In [4]:
from ATLAS.Analysis.Classification import *
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from ATLAS.Utils.analysisu import *
TMG_path = '/scratchdata1/MouseBrainAtlases_V7/'
animal = 'ASDM13'
# section = 'WTM01_7.8'
adata = anndata.read_h5ad(os.path.join(TMG_path,animal,'Layer','cell_layer.h5ad'))

adata.obs['ccf_z_half'] = np.abs(adata.obs['ccf_z'].copy()-5.71)
coords = torch.tensor((np.array(adata.obs[['ccf_x','ccf_y','ccf_z']])*1000/25).astype(int))-torch.tensor(np.array([ccf_x_min,0,0]))
coords[:,0] = torch.clamp(coords[:,0],0,readdata_torch.shape[0]-1)
coords[:,1] = torch.clamp(coords[:,1],0,readdata_torch.shape[1]-1)
coords[:,2] = torch.clamp(coords[:,2],0,readdata_torch.shape[2]-1)
adata.obs['ccf_region_index'] = readdata_torch[coords[:,0],coords[:,1],coords[:,2]].numpy()
for level in level_mapper.keys():
    adata.obs[f"ccf_region_label_{level}"] = adata.obs['ccf_region_index'].map(level_mapper[level])
    adata.obs[f"ccf_region_color_{level}"] = adata.obs[f"ccf_region_label_{level}"].map(color_mapper)
measured_adata = adata

# used_var = measured_adata.var.index
# adata = measured_adata[measured_adata.obs['Slice']=='WTM01_7.8']
# adata
measured_adata

AnnData object with n_obs × n_vars = 4043430 × 18
    obs: 'label', 'stage_x', 'stage_y', 'image_x', 'image_y', 'image_idx', 'size', 'section_index', 'dapi', 'sum', 'dataset', 'well', 'animal', 'processing', 'ccf_x', 'ccf_y', 'ccf_z', 'old_section_name', 'registration_path', 'section_name', 'Slice', 'in_large_comp', 'node_size', 'subclass', 'subclass_color', 'leiden', 'leiden_color', 'neuron', 'neuron_color', 'reference_neighbor_0', 'reference_neighbor_1', 'reference_neighbor_2', 'reference_neighbor_3', 'reference_neighbor_4', 'reference_neighbor_5', 'reference_neighbor_6', 'reference_neighbor_7', 'reference_neighbor_8', 'reference_neighbor_9', 'reference_neighbor_10', 'reference_neighbor_11', 'reference_neighbor_12', 'reference_neighbor_13', 'reference_neighbor_14', 'Correlation', 'subclass_id', 'rigid_ccf_z', 'rigid_ccf_y', 'rigid_deformation', 'ccf_z_half', 'ccf_region_index', 'ccf_region_label_0', 'ccf_region_color_0', 'ccf_region_label_1', 'ccf_region_color_1', 'ccf_region_label_2

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binned_statistic_dd
from scipy.ndimage import median_filter, gaussian_filter
from tqdm import tqdm
import pandas as pd

# --- Configuration ---
# This script assumes 'measured_adata' is loaded and is an AnnData object or similar
# structure with an '.obs' attribute that is a pandas DataFrame.
# e.g.,
# import anndata
# measured_adata = anndata.read_h5ad("path_to_your_anndata.h5ad")
#
# If 'measured_adata' is not available, you might need to uncomment and adapt
# the mock data generation section from the previous version for testing.


# --- Parameters ---
num_windows_to_visualize = 9 # Number of 2D slices to show
num_bins_slice_axis = 20     # Number of bins along the windowing_coord for 3D binning (e.g., 3 * num_windows_to_visualize)
num_bins_map_x = 200         # Number of bins for the map's x-axis (derived from ccf_z)
num_bins_map_y = 200         # Number of bins for the map's y-axis (derived from ccf_y)
min_cells_per_bin = 1        # Minimum number of cells required in a 3D bin to calculate a value

windowing_coord_name = 'ccf_x' # Column name for the axis to slice along

# Fixed range for the windowing coordinate for 3D binning
# These were hardcoded in your previous version.
# Ensure these values are appropriate for your 'ccf_x' data.
hardcoded_window_min_val = 4.5
hardcoded_window_max_val = 9.5

# --- Prepare Data for 3D Binning ---
print("Preparing data for 3D binning...")
obs_df = measured_adata.obs

# Coordinates for 3D binning and distances
windowing_coord_values = obs_df[windowing_coord_name].values
non_rigid_x_abs = np.abs(obs_df['ccf_z'].values - 5.71) # As per your definition for map's x-axis
non_rigid_y = -1 * obs_df['ccf_y'].values             # As per your definition for map's y-axis

# Rigid coordinates for distance calculation
rigid_x = obs_df['rigid_ccf_z'].values
rigid_y = -1 * obs_df['rigid_ccf_y'].values

# Original non-rigid coordinates for distance calculation (before abs and centering for map_x)
ccf_z_original = obs_df['ccf_z'].values
ccf_y_original = -1 * obs_df['ccf_y'].values


# Calculate the displacement distance for each cell
distances = np.sqrt((ccf_z_original - rigid_x)**2 + (ccf_y_original - rigid_y)**2)

# --- Determine Global Plot Limits and Binning Ranges ---
# For the map axes (x and y of the 2D plots)
plot_x_min = non_rigid_x_abs.min()
plot_x_max = non_rigid_x_abs.max()
plot_y_min = non_rigid_y.min()
plot_y_max = non_rigid_y.max()

# For the slicing axis (windowing_coord_name)
# Use hardcoded values if they are more restrictive than data range
actual_window_min = windowing_coord_values.min()
actual_window_max = windowing_coord_values.max()

# Effective range for the slice axis for 3D binning
# This ensures the binned statistic covers the specified range,
# but we only use data within this range if cells exist there.
binning_slice_axis_min = hardcoded_window_min_val
binning_slice_axis_max = hardcoded_window_max_val

print(f"Actual range for '{windowing_coord_name}': {actual_window_min:.2f} to {actual_window_max:.2f}")
print(f"Using hardcoded range for '{windowing_coord_name}' for 3D binning: {binning_slice_axis_min:.2f} to {binning_slice_axis_max:.2f}")


# Define sample points for binned_statistic_dd: (N, D) array, D=3
# Order: windowing_coord, map_x_coord, map_y_coord
sample_coords = np.vstack([windowing_coord_values, non_rigid_x_abs, non_rigid_y]).T

# Define bins and ranges for binned_statistic_dd
bins_3d = [num_bins_slice_axis, num_bins_map_x, num_bins_map_y]
ranges_3d = [
    [binning_slice_axis_min, binning_slice_axis_max],
    [plot_x_min, plot_x_max],
    [plot_y_min, plot_y_max]
]

# --- Perform 3D Binning ---
print(f"Performing 3D binning with bins: {bins_3d}...")
# Calculate mean distance in each 3D bin
mean_dist_3d_map, edges_3d, binnumber = binned_statistic_dd(
    sample_coords,
    values=distances,
    statistic='mean',
    bins=bins_3d,
    range=ranges_3d,
    expand_binnumbers=False # Kept False as per binned_statistic_2d default behavior for return
)

# Count cells in each 3D bin
counts_3d_map, _, _ = binned_statistic_dd(
    sample_coords,
    values=None, # Values not needed for 'count'
    statistic='count',
    bins=bins_3d,
    range=ranges_3d,
    expand_binnumbers=False
)
print("3D binning complete.")

# --- 3D Smoothing and Filtering (replicating user's logic for 3D) ---
print("Applying 3D smoothing and filtering...")
# Make copies to modify
processed_mean_dist_3d = mean_dist_3d_map.copy()
processed_counts_3d = counts_3d_map.copy()

# Store original NaN masks (where binned_statistic_dd found no cells or insufficient for mean)
nan_mask_mean_orig = np.isnan(processed_mean_dist_3d)
# counts_3d_map should not have NaNs if statistic is 'count' unless a bin is truly empty.
# For safety, handle potential NaNs in counts if they arise from upstream.
nan_mask_counts_orig = np.isnan(processed_counts_3d)
processed_counts_3d[nan_mask_counts_orig] = 0 # Ensure counts are numeric for filtering


# Median filter part (isotropic filter size 5, as in user's 2D code)
# Apply to a version where NaNs are temporarily filled (e.g., with 0)
temp_mean_for_median = np.where(nan_mask_mean_orig, 0, processed_mean_dist_3d)
filtered_mean_median = median_filter(temp_mean_for_median, size=5)
# Fill original NaN locations in processed_mean_dist_3d with these filtered values
processed_mean_dist_3d[nan_mask_mean_orig] = filtered_mean_median[nan_mask_mean_orig]

temp_counts_for_median = np.where(nan_mask_counts_orig, 0, processed_counts_3d)
filtered_counts_median = median_filter(temp_counts_for_median, size=5)
processed_counts_3d[nan_mask_counts_orig] = filtered_counts_median[nan_mask_counts_orig]


# Gaussian filter part (isotropic sigma 1, as in user's 2D code)
# Apply to non-NaN parts (after median filter might have filled some NaNs)
# Re-evaluate non-NaN mask for mean. Counts should be all numeric now.
not_nan_mask_mean_after_median = ~np.isnan(processed_mean_dist_3d)

# Filter a version where remaining NaNs (if any) are temporarily 0
temp_mean_for_gaussian = np.where(not_nan_mask_mean_after_median, processed_mean_dist_3d, 0)
filtered_mean_gaussian = gaussian_filter(temp_mean_for_gaussian, sigma=1)
# Update original non-NaN locations
processed_mean_dist_3d[not_nan_mask_mean_after_median] = filtered_mean_gaussian[not_nan_mask_mean_after_median]

# Gaussian filter for counts (should be all numeric, but for consistency)
temp_counts_for_gaussian = processed_counts_3d # Assuming counts are now fully numeric from median step
filtered_counts_gaussian = gaussian_filter(temp_counts_for_gaussian, sigma=1)
# Update counts (here, all elements as it should be dense)
processed_counts_3d = filtered_counts_gaussian


# Final thresholding based on min_cells_per_bin using the (smoothed) counts
final_mean_dist_3d_map = processed_mean_dist_3d
final_mean_dist_3d_map[processed_counts_3d < min_cells_per_bin] = np.nan
print("3D smoothing and filtering complete.")

# --- Initialize Figure for 2D Slice Visualization ---
fig, axes = plt.subplots(3, 3, figsize=(15, 15), dpi=300)
axes = axes.ravel()

vmin_plot = 0
vmax_plot = 1 # Adjust as needed

# Determine slice indices to visualize from the 3D binned data
slice_indices = np.linspace(0, num_bins_slice_axis - 1, num_windows_to_visualize).astype(int)
# Ensure unique indices if num_bins_slice_axis is small
slice_indices = np.unique(slice_indices) 
if len(slice_indices) < num_windows_to_visualize and num_bins_slice_axis >= num_windows_to_visualize:
    # Fallback if linspace + unique resulted in fewer due to rounding with small num_bins_slice_axis
    slice_indices = np.round(np.linspace(0, num_bins_slice_axis - 1, num_windows_to_visualize)).astype(int)
    slice_indices = np.unique(slice_indices) # May still result in fewer if num_bins_slice_axis is too small


print(f"Visualizing {len(slice_indices)} slices from 3D data...")

# --- Loop through selected slices for visualization ---
for i, ax_idx in enumerate(range(len(slice_indices))): # Iterate up to number of available unique slices
    if ax_idx >= len(axes): # Should not happen if num_windows_to_visualize is 9
        break
    
    ax = axes[ax_idx]
    current_slice_index_in_3d = slice_indices[ax_idx]

    # Extract the 2D slice from the 3D map
    # final_mean_dist_3d_map has shape (num_bins_slice_axis, num_bins_map_x, num_bins_map_y)
    slice_2d_map = final_mean_dist_3d_map[current_slice_index_in_3d, :, :]

    # Determine the windowing_coord range for this slice's title
    slice_axis_bin_centers = (edges_3d[0][:-1] + edges_3d[0][1:]) / 2
    slice_start_coord = edges_3d[0][current_slice_index_in_3d]
    slice_end_coord = edges_3d[0][current_slice_index_in_3d + 1]

    im = ax.imshow(slice_2d_map.T, aspect='auto', origin='lower',
                   extent=[plot_x_min, plot_x_max, plot_y_min, plot_y_max],
                   cmap='inferno', vmin=vmin_plot, vmax=vmax_plot)
    if i == len(slice_indices)-1:
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.05) # Adjust size and pad as needed
        plt.colorbar(im, cax=cax)
    # plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.046, pad=0.04)

    ax.axis('off')
    ax.grid(False)
    val = (slice_start_coord+slice_end_coord)/2
    # round val to nearest 0.25
    val = round(val * 4) / 4  # Round to nearest 0.25
    ax.set_title(f"{windowing_coord_name}: {val}", fontsize=16)
    
    ax.set_xlim(plot_x_min, plot_x_max)
    ax.set_ylim(plot_y_min, plot_y_max)

# If fewer than 9 unique slices were generated, turn off remaining axes
for j in range(len(slice_indices), len(axes)):
    axes[j].axis('off')
    axes[j].text(0.5, 0.5, 'No slice data', horizontalalignment='center', verticalalignment='center', transform=axes[j].transAxes)


# --- Final Figure Adjustments ---
plt.tight_layout(rect=[0, 0, 1, 0.96]) # Adjust rect to make space for suptitle
# plt.suptitle(f"Region-Level Registration Difference (3D Binned)\nSlices along {windowing_coord_name}", fontsize=16, y=0.99)

# To save the figure:
plt.savefig('/scratchdata1/MouseBrainAtlases_V7/AnalysisNotebooks/Panels/ASD_3D_Binned_Region_Level_Difference.pdf', dpi=500)
plt.show()

print("Script finished.")

Preparing data for 3D binning...
Actual range for 'ccf_x': 4.20 to 13.20
Using hardcoded range for 'ccf_x' for 3D binning: 4.50 to 9.50
Performing 3D binning with bins: [20, 200, 200]...
3D binning complete.
Applying 3D smoothing and filtering...
3D smoothing and filtering complete.
Visualizing 9 slices from 3D data...


INFO:fontTools.subset:maxp pruned
INFO:fontTools.subset:cmap pruned
INFO:fontTools.subset:kern dropped
INFO:fontTools.subset:post pruned
INFO:fontTools.subset:FFTM dropped
INFO:fontTools.subset:GPOS pruned
INFO:fontTools.subset:GSUB pruned
INFO:fontTools.subset:name pruned
INFO:fontTools.subset:glyf pruned
INFO:fontTools.subset:Added gid0 to subset
INFO:fontTools.subset:Added first four glyphs to subset
INFO:fontTools.subset:Closing glyph list over 'GSUB': 19 glyphs before
INFO:fontTools.subset:Glyph names: ['.notdef', '.null', 'c', 'colon', 'eight', 'f', 'five', 'four', 'nine', 'nonmarkingreturn', 'one', 'period', 'seven', 'six', 'space', 'two', 'underscore', 'x', 'zero']
INFO:fontTools.subset:Glyph IDs:   [0, 1, 2, 3, 17, 19, 20, 21, 23, 24, 25, 26, 27, 28, 29, 66, 70, 73, 91]
INFO:fontTools.subset:Closed glyph list over 'GSUB': 20 glyphs after
INFO:fontTools.subset:Glyph names: ['.notdef', '.null', 'c', 'colon', 'eight', 'f', 'five', 'four', 'nine', 'nonmarkingreturn', 'one', 'perio

Script finished.


In [6]:
from ATLAS.Analysis.Classification import *
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from ATLAS.Utils.analysisu import *
TMG_path = '/scratchdata1/MouseBrainAtlases_V7/'
animal = 'WTM11'
# section = 'WTM01_7.8'
adata = anndata.read_h5ad(os.path.join(TMG_path,animal,'Layer','cell_layer.h5ad'))

adata.obs['ccf_z_half'] = np.abs(adata.obs['ccf_z'].copy()-5.71)
coords = torch.tensor((np.array(adata.obs[['ccf_x','ccf_y','ccf_z']])*1000/25).astype(int))-torch.tensor(np.array([ccf_x_min,0,0]))
coords[:,0] = torch.clamp(coords[:,0],0,readdata_torch.shape[0]-1)
coords[:,1] = torch.clamp(coords[:,1],0,readdata_torch.shape[1]-1)
coords[:,2] = torch.clamp(coords[:,2],0,readdata_torch.shape[2]-1)
adata.obs['ccf_region_index'] = readdata_torch[coords[:,0],coords[:,1],coords[:,2]].numpy()
for level in level_mapper.keys():
    adata.obs[f"ccf_region_label_{level}"] = adata.obs['ccf_region_index'].map(level_mapper[level])
    adata.obs[f"ccf_region_color_{level}"] = adata.obs[f"ccf_region_label_{level}"].map(color_mapper)
measured_adata = adata

# used_var = measured_adata.var.index
# adata = measured_adata[measured_adata.obs['Slice']=='WTM01_7.8']
# adata
measured_adata

AnnData object with n_obs × n_vars = 4214349 × 18
    obs: 'label', 'stage_x', 'stage_y', 'image_x', 'image_y', 'image_idx', 'size', 'section_index', 'dapi', 'sum', 'dataset', 'well', 'animal', 'processing', 'ccf_x', 'ccf_y', 'ccf_z', 'old_section_name', 'registration_path', 'section_name', 'Slice', 'in_large_comp', 'node_size', 'subclass', 'subclass_color', 'leiden', 'leiden_color', 'neuron', 'neuron_color', 'reference_neighbor_0', 'reference_neighbor_1', 'reference_neighbor_2', 'reference_neighbor_3', 'reference_neighbor_4', 'reference_neighbor_5', 'reference_neighbor_6', 'reference_neighbor_7', 'reference_neighbor_8', 'reference_neighbor_9', 'reference_neighbor_10', 'reference_neighbor_11', 'reference_neighbor_12', 'reference_neighbor_13', 'reference_neighbor_14', 'Correlation', 'subclass_id', 'rigid_ccf_z', 'rigid_ccf_y', 'rigid_deformation', 'ccf_z_half', 'ccf_region_index', 'ccf_region_label_0', 'ccf_region_color_0', 'ccf_region_label_1', 'ccf_region_color_1', 'ccf_region_label_2

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binned_statistic_dd
from scipy.ndimage import median_filter, gaussian_filter
from tqdm import tqdm
import pandas as pd

# --- Configuration ---
# This script assumes 'measured_adata' is loaded and is an AnnData object or similar
# structure with an '.obs' attribute that is a pandas DataFrame.
# e.g.,
# import anndata
# measured_adata = anndata.read_h5ad("path_to_your_anndata.h5ad")
#
# If 'measured_adata' is not available, you might need to uncomment and adapt
# the mock data generation section from the previous version for testing.


# --- Parameters ---
num_windows_to_visualize = 9 # Number of 2D slices to show
num_bins_slice_axis = 20     # Number of bins along the windowing_coord for 3D binning (e.g., 3 * num_windows_to_visualize)
num_bins_map_x = 200         # Number of bins for the map's x-axis (derived from ccf_z)
num_bins_map_y = 200         # Number of bins for the map's y-axis (derived from ccf_y)
min_cells_per_bin = 1        # Minimum number of cells required in a 3D bin to calculate a value

windowing_coord_name = 'ccf_x' # Column name for the axis to slice along

# Fixed range for the windowing coordinate for 3D binning
# These were hardcoded in your previous version.
# Ensure these values are appropriate for your 'ccf_x' data.
hardcoded_window_min_val = 4.5
hardcoded_window_max_val = 9.5

# --- Prepare Data for 3D Binning ---
print("Preparing data for 3D binning...")
obs_df = measured_adata.obs

# Coordinates for 3D binning and distances
windowing_coord_values = obs_df[windowing_coord_name].values
non_rigid_x_abs = np.abs(obs_df['ccf_z'].values - 5.71) # As per your definition for map's x-axis
non_rigid_y = -1 * obs_df['ccf_y'].values             # As per your definition for map's y-axis

# Rigid coordinates for distance calculation
rigid_x = obs_df['rigid_ccf_z'].values
rigid_y = -1 * obs_df['rigid_ccf_y'].values

# Original non-rigid coordinates for distance calculation (before abs and centering for map_x)
ccf_z_original = obs_df['ccf_z'].values
ccf_y_original = -1 * obs_df['ccf_y'].values


# Calculate the displacement distance for each cell
distances = np.sqrt((ccf_z_original - rigid_x)**2 + (ccf_y_original - rigid_y)**2)

# --- Determine Global Plot Limits and Binning Ranges ---
# For the map axes (x and y of the 2D plots)
plot_x_min = non_rigid_x_abs.min()
plot_x_max = non_rigid_x_abs.max()
plot_y_min = non_rigid_y.min()
plot_y_max = non_rigid_y.max()

# For the slicing axis (windowing_coord_name)
# Use hardcoded values if they are more restrictive than data range
actual_window_min = windowing_coord_values.min()
actual_window_max = windowing_coord_values.max()

# Effective range for the slice axis for 3D binning
# This ensures the binned statistic covers the specified range,
# but we only use data within this range if cells exist there.
binning_slice_axis_min = hardcoded_window_min_val
binning_slice_axis_max = hardcoded_window_max_val

print(f"Actual range for '{windowing_coord_name}': {actual_window_min:.2f} to {actual_window_max:.2f}")
print(f"Using hardcoded range for '{windowing_coord_name}' for 3D binning: {binning_slice_axis_min:.2f} to {binning_slice_axis_max:.2f}")


# Define sample points for binned_statistic_dd: (N, D) array, D=3
# Order: windowing_coord, map_x_coord, map_y_coord
sample_coords = np.vstack([windowing_coord_values, non_rigid_x_abs, non_rigid_y]).T

# Define bins and ranges for binned_statistic_dd
bins_3d = [num_bins_slice_axis, num_bins_map_x, num_bins_map_y]
ranges_3d = [
    [binning_slice_axis_min, binning_slice_axis_max],
    [plot_x_min, plot_x_max],
    [plot_y_min, plot_y_max]
]

# --- Perform 3D Binning ---
print(f"Performing 3D binning with bins: {bins_3d}...")
# Calculate mean distance in each 3D bin
mean_dist_3d_map, edges_3d, binnumber = binned_statistic_dd(
    sample_coords,
    values=distances,
    statistic='mean',
    bins=bins_3d,
    range=ranges_3d,
    expand_binnumbers=False # Kept False as per binned_statistic_2d default behavior for return
)

# Count cells in each 3D bin
counts_3d_map, _, _ = binned_statistic_dd(
    sample_coords,
    values=None, # Values not needed for 'count'
    statistic='count',
    bins=bins_3d,
    range=ranges_3d,
    expand_binnumbers=False
)
print("3D binning complete.")

# --- 3D Smoothing and Filtering (replicating user's logic for 3D) ---
print("Applying 3D smoothing and filtering...")
# Make copies to modify
processed_mean_dist_3d = mean_dist_3d_map.copy()
processed_counts_3d = counts_3d_map.copy()

# Store original NaN masks (where binned_statistic_dd found no cells or insufficient for mean)
nan_mask_mean_orig = np.isnan(processed_mean_dist_3d)
# counts_3d_map should not have NaNs if statistic is 'count' unless a bin is truly empty.
# For safety, handle potential NaNs in counts if they arise from upstream.
nan_mask_counts_orig = np.isnan(processed_counts_3d)
processed_counts_3d[nan_mask_counts_orig] = 0 # Ensure counts are numeric for filtering


# Median filter part (isotropic filter size 5, as in user's 2D code)
# Apply to a version where NaNs are temporarily filled (e.g., with 0)
temp_mean_for_median = np.where(nan_mask_mean_orig, 0, processed_mean_dist_3d)
filtered_mean_median = median_filter(temp_mean_for_median, size=5)
# Fill original NaN locations in processed_mean_dist_3d with these filtered values
processed_mean_dist_3d[nan_mask_mean_orig] = filtered_mean_median[nan_mask_mean_orig]

temp_counts_for_median = np.where(nan_mask_counts_orig, 0, processed_counts_3d)
filtered_counts_median = median_filter(temp_counts_for_median, size=5)
processed_counts_3d[nan_mask_counts_orig] = filtered_counts_median[nan_mask_counts_orig]


# Gaussian filter part (isotropic sigma 1, as in user's 2D code)
# Apply to non-NaN parts (after median filter might have filled some NaNs)
# Re-evaluate non-NaN mask for mean. Counts should be all numeric now.
not_nan_mask_mean_after_median = ~np.isnan(processed_mean_dist_3d)

# Filter a version where remaining NaNs (if any) are temporarily 0
temp_mean_for_gaussian = np.where(not_nan_mask_mean_after_median, processed_mean_dist_3d, 0)
filtered_mean_gaussian = gaussian_filter(temp_mean_for_gaussian, sigma=1)
# Update original non-NaN locations
processed_mean_dist_3d[not_nan_mask_mean_after_median] = filtered_mean_gaussian[not_nan_mask_mean_after_median]

# Gaussian filter for counts (should be all numeric, but for consistency)
temp_counts_for_gaussian = processed_counts_3d # Assuming counts are now fully numeric from median step
filtered_counts_gaussian = gaussian_filter(temp_counts_for_gaussian, sigma=1)
# Update counts (here, all elements as it should be dense)
processed_counts_3d = filtered_counts_gaussian


# Final thresholding based on min_cells_per_bin using the (smoothed) counts
final_mean_dist_3d_map = processed_mean_dist_3d
final_mean_dist_3d_map[processed_counts_3d < min_cells_per_bin] = np.nan
print("3D smoothing and filtering complete.")

# --- Initialize Figure for 2D Slice Visualization ---
fig, axes = plt.subplots(3, 3, figsize=(15, 15), dpi=300)
axes = axes.ravel()

vmin_plot = 0
vmax_plot = 1 # Adjust as needed

# Determine slice indices to visualize from the 3D binned data
slice_indices = np.linspace(0, num_bins_slice_axis - 1, num_windows_to_visualize).astype(int)
# Ensure unique indices if num_bins_slice_axis is small
slice_indices = np.unique(slice_indices) 
if len(slice_indices) < num_windows_to_visualize and num_bins_slice_axis >= num_windows_to_visualize:
    # Fallback if linspace + unique resulted in fewer due to rounding with small num_bins_slice_axis
    slice_indices = np.round(np.linspace(0, num_bins_slice_axis - 1, num_windows_to_visualize)).astype(int)
    slice_indices = np.unique(slice_indices) # May still result in fewer if num_bins_slice_axis is too small


print(f"Visualizing {len(slice_indices)} slices from 3D data...")

# --- Loop through selected slices for visualization ---
for i, ax_idx in enumerate(range(len(slice_indices))): # Iterate up to number of available unique slices
    if ax_idx >= len(axes): # Should not happen if num_windows_to_visualize is 9
        break
    
    ax = axes[ax_idx]
    current_slice_index_in_3d = slice_indices[ax_idx]

    # Extract the 2D slice from the 3D map
    # final_mean_dist_3d_map has shape (num_bins_slice_axis, num_bins_map_x, num_bins_map_y)
    slice_2d_map = final_mean_dist_3d_map[current_slice_index_in_3d, :, :]

    # Determine the windowing_coord range for this slice's title
    slice_axis_bin_centers = (edges_3d[0][:-1] + edges_3d[0][1:]) / 2
    slice_start_coord = edges_3d[0][current_slice_index_in_3d]
    slice_end_coord = edges_3d[0][current_slice_index_in_3d + 1]

    im = ax.imshow(slice_2d_map.T, aspect='auto', origin='lower',
                   extent=[plot_x_min, plot_x_max, plot_y_min, plot_y_max],
                   cmap='inferno', vmin=vmin_plot, vmax=vmax_plot)
    if i == len(slice_indices)-1:
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.05) # Adjust size and pad as needed
        plt.colorbar(im, cax=cax)
        # plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.046, pad=0.04)

    ax.axis('off')
    ax.grid(False)
    val = (slice_start_coord+slice_end_coord)/2
    # round val to nearest 0.25
    val = round(val * 4) / 4  # Round to nearest 0.25
    ax.set_title(f"{windowing_coord_name}: {val}", fontsize=16)
    
    ax.set_xlim(plot_x_min, plot_x_max)
    ax.set_ylim(plot_y_min, plot_y_max)

# If fewer than 9 unique slices were generated, turn off remaining axes
for j in range(len(slice_indices), len(axes)):
    axes[j].axis('off')
    axes[j].text(0.5, 0.5, 'No slice data', horizontalalignment='center', verticalalignment='center', transform=axes[j].transAxes)


# --- Final Figure Adjustments ---
plt.tight_layout(rect=[0, 0, 1, 0.96]) # Adjust rect to make space for suptitle
# plt.suptitle(f"Region-Level Registration Difference (3D Binned)\nSlices along {windowing_coord_name}", fontsize=16, y=0.99)

# To save the figure:
# plt.savefig('/path_to_your_output/3D_Binned_Region_Level_Difference.png', dpi=300, bbox_inches='tight')
plt.savefig('/scratchdata1/MouseBrainAtlases_V7/AnalysisNotebooks/Panels/WT_3D_Binned_Region_Level_Difference.pdf', dpi=500)
plt.show()

print("Script finished.")

Preparing data for 3D binning...


Actual range for 'ccf_x': 4.00 to 13.20
Using hardcoded range for 'ccf_x' for 3D binning: 4.50 to 9.50
Performing 3D binning with bins: [20, 200, 200]...
3D binning complete.
Applying 3D smoothing and filtering...
3D smoothing and filtering complete.
Visualizing 9 slices from 3D data...


INFO:fontTools.subset:maxp pruned
INFO:fontTools.subset:cmap pruned
INFO:fontTools.subset:kern dropped
INFO:fontTools.subset:post pruned
INFO:fontTools.subset:FFTM dropped
INFO:fontTools.subset:GPOS pruned
INFO:fontTools.subset:GSUB pruned
INFO:fontTools.subset:name pruned
INFO:fontTools.subset:glyf pruned
INFO:fontTools.subset:Added gid0 to subset
INFO:fontTools.subset:Added first four glyphs to subset
INFO:fontTools.subset:Closing glyph list over 'GSUB': 19 glyphs before
INFO:fontTools.subset:Glyph names: ['.notdef', '.null', 'c', 'colon', 'eight', 'f', 'five', 'four', 'nine', 'nonmarkingreturn', 'one', 'period', 'seven', 'six', 'space', 'two', 'underscore', 'x', 'zero']
INFO:fontTools.subset:Glyph IDs:   [0, 1, 2, 3, 17, 19, 20, 21, 23, 24, 25, 26, 27, 28, 29, 66, 70, 73, 91]
INFO:fontTools.subset:Closed glyph list over 'GSUB': 20 glyphs after
INFO:fontTools.subset:Glyph names: ['.notdef', '.null', 'c', 'colon', 'eight', 'f', 'five', 'four', 'nine', 'nonmarkingreturn', 'one', 'perio

Script finished.
